# NB10B — pHash + SSIM + exact-pixel ablation

Bản này bật exact-pixel shortcut để làm ablation. Exact match được report riêng, nhưng pHash/SSIM thresholds vẫn giữ nguyên.

Decision states trước manual review chỉ có **DUPLICATE / MANUAL_REVIEW / NON_DUPLICATE**.

- `pHash <= 4` và `SSIM >= 0.92` → `DUPLICATE`
- `pHash <= 4` và `0.90 <= SSIM < 0.92` → `MANUAL_REVIEW`
- còn lại → `NON_DUPLICATE`

Manual review cuối cùng chỉ chọn **DUPLICATE** hoặc **NON_DUPLICATE**. ID collision và dHash không tham gia decision path.

> Lưu ý implementation: notebook dùng standard 64-bit DCT pHash và SSIM trên grayscale 256×256. Threshold 4 / 0.90 / 0.92 chỉ nên được coi là official nếu implementation này khớp implementation dùng lúc calibration.


## 1. Runtime + branch riêng


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

REPO_URL = 'https://github.com/ThinhTran2208/opisoverated.git'
BRANCH = 'feat/evaluation3-phash-ssim-overlap-v2'
DEFAULT_REPO_DIR = Path('/content/opisoverated-e3-phash-ssim-v2')
explicit_root = os.environ.get('FASHION_PROJECT_ROOT')
REPO_ROOT = Path(explicit_root).expanduser().resolve() if explicit_root else DEFAULT_REPO_DIR

def run_git(*args, cwd=None):
    return subprocess.run(['git', '-c', 'http.version=HTTP/1.1', *args], cwd=cwd, check=True, text=True)

if not (REPO_ROOT / '.git').is_dir():
    if REPO_ROOT.exists():
        raise FileExistsError(f'{REPO_ROOT} tồn tại nhưng không phải Git repo.')
    run_git('clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT))
else:
    run_git('fetch', 'origin', BRANCH, cwd=REPO_ROOT)
    run_git('switch', BRANCH, cwd=REPO_ROOT)
    run_git('pull', '--ff-only', 'origin', BRANCH, cwd=REPO_ROOT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements-evaluation.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.evaluation3_phash_ssim import prepare_overlap_audit, finalize_overlap_audit

USE_EXACT_PIXEL = True
print('REPO_ROOT:', REPO_ROOT)
print('USE_EXACT_PIXEL:', USE_EXACT_PIXEL)


## 2. Resolve input trên Google Drive


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
E3_DIR = DRIVE_ROOT / 'EVALUATION3'
E3_ARCHIVE = E3_DIR / 'outfit.zip'
CMT_FILE = E3_DIR / 'Cmt_ALL_20190325.xlsx'
ATTRIBUTE_FILE = E3_DIR / 'Attribute_ALL_UBSGsimple.xlsx'
SCORER_DIR = DRIVE_ROOT / 'scorer_ready_v2'
OUTPUT_BASE = DRIVE_ROOT / 'evaluation3_overlap_phash_ssim_v2'
OUTPUT_DIR = OUTPUT_BASE / 'with_exact_pixel'
LOCAL_E3_DIR = Path('/content/evaluation3_phash_ssim_v2')
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def contains_required_e3_image(root):
    if not root.is_dir():
        return False
    return any(
        p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES and p.stem.upper() in {'U','B','S','G'}
        for p in root.rglob('*')
    )

def resolve_e3_root():
    for candidate in (E3_DIR / 'outfit', E3_DIR / 'Outfits'):
        if contains_required_e3_image(candidate):
            return candidate
    if E3_ARCHIVE.is_file():
        if not contains_required_e3_image(LOCAL_E3_DIR):
            if LOCAL_E3_DIR.exists():
                shutil.rmtree(LOCAL_E3_DIR)
            LOCAL_E3_DIR.mkdir(parents=True, exist_ok=True)
            print('Unpacking EVALUATION3 archive to local Colab disk...')
            shutil.unpack_archive(str(E3_ARCHIVE), str(LOCAL_E3_DIR), format='zip')
        for candidate in (LOCAL_E3_DIR / 'outfit', LOCAL_E3_DIR / 'Outfits', LOCAL_E3_DIR):
            if contains_required_e3_image(candidate):
                return candidate
    return E3_DIR / 'outfit'

E3_ROOT = resolve_e3_root()
TRAIN_FILE = SCORER_DIR / 'scorer_ready_v2_train.jsonl'
VALID_FILE = SCORER_DIR / 'scorer_ready_v2_valid.jsonl'
TEST_FILE = SCORER_DIR / 'scorer_ready_v2_test.jsonl'
required = {
    'E3 images': E3_ROOT,
    'Cmt workbook': CMT_FILE,
    'Attribute workbook': ATTRIBUTE_FILE,
    'Scorer train': TRAIN_FILE,
    'Scorer valid': VALID_FILE,
    'Scorer test': TEST_FILE,
}
missing = []
for name, path in required.items():
    ok = contains_required_e3_image(path) if name == 'E3 images' else path.is_file()
    if not ok:
        missing.append(f'{name}: {path}')
INPUTS_READY = not missing
print('E3_ROOT:', E3_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
if INPUTS_READY:
    print('INPUTS: OK')
else:
    print('INPUTS: CHƯA ĐỦ')
    for value in missing:
        print(' -', value)


## 3. Prepare overlap + tạo manual-review queue


In [ ]:
summary = None
prepare_paths = {}
if not INPUTS_READY:
    print('SKIPPED')
else:
    development_splits = {
        'train': TRAIN_FILE,
        'valid': VALID_FILE,
        'test': TEST_FILE,
    }
    summary, prepare_paths = prepare_overlap_audit(
        evaluation3_root=E3_ROOT,
        development_split_paths=development_splits,
        output_dir=OUTPUT_DIR,
        polyvore_hf_dataset='codewaly/polyvore1000',
        annotations_path=CMT_FILE,
        annotation_sheet='CMT',
        metadata_path=ATTRIBUTE_FILE,
        metadata_sheet='Num',
        model_development_splits={'train', 'valid'},
        use_exact_pixel=USE_EXACT_PIXEL,
        phash_threshold=4,
        ssim_auto_threshold=0.92,
        ssim_manual_lower_bound=0.90,
        ssim_size=256,
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print('\nOutputs:')
    for name, path in prepare_paths.items():
        print(f'- {name}: {path}')


## 4. Manual review

Nếu `manual_review_pairs > 0`:

1. Mở `evaluation3_manual_review_BLIND.html` / folder `manual_review_previews` để xem từng pair.
2. Mở **`evaluation3_manual_review_BLIND.xlsx`** trong Drive.
3. Cột `human_label` chỉ chọn `DUPLICATE` hoặc `NON_DUPLICATE`.
4. Không cần `UNCERTAIN`, `SKIP`, hay `SAME_PRODUCT...`.
5. `evaluation3_manual_review_KEY.csv` chứa pHash/SSIM/split để audit sau; đừng nhìn KEY trong lúc label nếu muốn giữ blind review.

Không có clean manifest official trước khi manual queue được resolve hết.


## 5. Finalize sau khi review xong


In [ ]:
final_summary = None
final_paths = {}
if not INPUTS_READY:
    print('SKIPPED')
else:
    manual_xlsx = OUTPUT_DIR / 'evaluation3_manual_review_BLIND.xlsx'
    final_summary, final_paths = finalize_overlap_audit(
        output_dir=OUTPUT_DIR,
        manual_labels_path=manual_xlsx,
        model_development_splits={'train', 'valid'},
    )
    print(json.dumps(final_summary, ensure_ascii=False, indent=2))
    print('\nFinal outputs:')
    for name, path in final_paths.items():
        print(f'- {name}: {path}')
    if not final_summary['official_clean_manifests_ready']:
        print('\nCHƯA OFFICIAL: resolve hết manual labels và mọi missing image rồi chạy lại cell này.')
    else:
        print('\nPASS: model_clean / strict_clean đã sẵn sàng cho scorer evaluation.')
